<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/8.%20Agentic%20AI%20Systems/18.%20Rapid%20Prototyping%20%26%20Low-Code%20Tools/L17_%20low-code-academic_research_agent_learner.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Academic Research Agent

This notebook explores the building blocks of a research agent: language models, search tools, memory, prompts, and agent execution. The result is an agent that combines academic papers and current web context into a structured research briefing.


## Concepts

- **Language model**: interprets the research question and coordinates the workflow.
- **Tools**: provide access to Arxiv, web search, and research summarization.
- **Memory**: retains prior messages within a research session.
- **Prompt**: defines the scope and format of the research briefing.
- **Agent executor**: runs the loop between the model and its tools.


## Environment

The notebook uses Groq for the language model, Arxiv for scholarly papers, and DuckDuckGo for supplementary context. A Groq API key is requested locally at runtime and is not stored in the notebook.


In [ ]:
!pip install -q langchain-classic langchain-groq langchain-community arxiv duckduckgo-search xmltodict


In [ ]:
import os
import getpass
import arxiv as arxiv_lib

from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool


## Language model

The model is configured with a low temperature so that research briefings tend toward consistent, focused output.


In [ ]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3,
    max_tokens=1024,
)


## Research tools

The agent has three distinct capabilities: finding papers, collecting recent context, and organizing gathered material. Tool descriptions are part of the interface the model uses when selecting a capability.


In [ ]:
@tool
def arxiv(query: str) -> str:
    """Find relevant academic papers on Arxiv. Returns titles, authors, publication dates, and abstracts."""
    client = arxiv_lib.Client()
    search = arxiv_lib.Search(
        query=query,
        max_results=5,
        sort_by=arxiv_lib.SortCriterion.Relevance,
    )
    results = []
    for paper in client.results(search):
        results.append(
            f"Title: {paper.title}\n"
            f"Authors: {', '.join(author.name for author in paper.authors)}\n"
            f"Published: {paper.published.date()}\n"
            f"Summary: {paper.summary[:200]}...\n"
        )
    return "\n---\n".join(results) if results else "No results found."


ddg_search = DuckDuckGoSearchRun()

def web_search_capped(query: str) -> str:
    return ddg_search.run(query)[:600]


web_search_tool = Tool(
    name="web_search",
    func=web_search_capped,
    description=(
        "Find current information and practical context beyond academic papers, "
        "including recent developments and industry applications."
    ),
)


def summarize_research(text: str) -> str:
    summary_prompt = ChatPromptTemplate.from_template(
        """Analyze the research information below and organize it into:

        1. Key findings
        2. Common themes
        3. Open questions
        4. Practical implications

        Research information:
        {text}
        """
    )
    return (summary_prompt | llm).invoke({"text": text[:1500]}).content


summarizer_tool = Tool(
    name="summarize_research",
    func=summarize_research,
    description="Organize collected research information into a concise structured summary.",
)

tools = [arxiv, web_search_tool, summarizer_tool]


## Memory and agent prompt

Conversation memory carries context from one question to the next. The prompt specifies the research workflow and the final briefing structure.


In [ ]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)

system_prompt = """You are an academic research assistant.

For each topic, search Arxiv for relevant papers, examine their themes and findings,
then gather supplementary web context when it adds value. Organize the final response as:
- Overview
- Key papers
- Web insights
- Synthesis
- Further research

Broaden a sparse Arxiv search with related terms. Clearly distinguish evidence from
academic papers from context found on the web."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])


## Agent assembly

The executor gives the model access to the tools and memory. A bounded number of iterations prevents an unbounded tool loop.


In [ ]:
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=10,
)


## Research briefing

The query below examines attention mechanisms and transformer architectures. It can be replaced with another research topic.


In [ ]:
query = "Impact of attention mechanisms on natural language processing, focusing on transformer architectures"
result = agent_executor.invoke({"input": query})
print(result["output"])


## Follow-up research

The follow-up uses the same session memory, allowing the agent to build on earlier findings.


In [ ]:
follow_up_query = "Which papers from the earlier results address efficiency improvements in attention mechanisms? Include recent developments."
result2 = agent_executor.invoke({"input": follow_up_query})
print(result2["output"])


## Architecture variations

Restricting an agent's toolset creates a focused variant. This example performs a paper-only scan.


In [ ]:
quick_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a research scanner. Find the three most relevant Arxiv papers
    for a topic and give a brief summary of each paper's contribution."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

quick_executor = AgentExecutor(
    agent=create_tool_calling_agent(llm=llm, tools=[arxiv], prompt=quick_prompt),
    tools=[arxiv],
    memory=ConversationBufferMemory(memory_key="chat_history", return_messages=True),
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=5,
)

quick_result = quick_executor.invoke({
    "input": "Recent papers on retrieval-augmented generation"
})
print(quick_result["output"])


## Key ideas

An academic research agent combines retrieval, reasoning, and synthesis. Tool descriptions shape tool selection, memory supports multi-turn research, and the prompt defines the scope and reporting format. The same components can be reconfigured for broad research briefings or narrower paper scans.
